In [1]:
## Import packages
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from torch.distributions import Beta
from IPython.display import clear_output




In [2]:
# Parameter Values

# Preferences
gamma = 2.1
rho = 0.05
kappa = 3.0
a_lb = 1.0
a_min = 1e-2
a_max = 20.0


# Idiosyncratic Shocks
lambda1 = 0.4
lambda2 = 0.4
lambdas = torch.tensor([lambda1,lambda2]) 

n1 = 0.3
n2 = 1 + (lambda2/lambda1)*(1-n1)
n_vals = torch.tensor([n1, n2], dtype=torch.float32)
L_bar = 1 


# Aggregate Shocks
sig2 = 0.01
Zbar = 0.0
eta = 0.5
z_min = math.log(0.96)
z_max = math.log(1.04)


# Market Clearing
alpha = 1/3
delta = 0.1

N_pop = 41


# Steady State Values (Obtained by Solving Aiyagari Model via FD) 
r_eq = 0.012
r_min = -0.05 
r_max = 0.05

w_eq = 1.15





In [ ]:
# Steady State Distribution
df = pd.read_csv("stat_dist_fd_Seung.csv")
A_grid = torch.tensor(df["x1"].values, dtype=torch.float32)
g1 = torch.tensor(df["x2"].values, dtype=torch.float32)
g2 = torch.tensor(df["x3"].values, dtype=torch.float32)
g = g1 + g2 

k_ss = torch.dot(A_grid,g) 
var_a= torch.dot(A_grid**2, g) - k_ss**2

dist = torch.distributions.Categorical(probs=g)




In [4]:
## Target Variance for Extended Beta Distribution
def beta_pars(k, v, a_min, a_max):
    cons = (a_max - k) / (a_max - a_min)
    max_var = (a_max - k) * (k - a_min)
    v_eff = torch.minimum(v, max_var - 1e-4)
    kernel = torch.clamp_min((a_max - k) * (k - a_min) / v_eff - 1.0 , 1e-8) 
    beta_beta = cons * kernel
    alpha_beta = (1.0 - cons) * kernel
    return alpha_beta, beta_beta



In [5]:
## Sampling and Equilibrium
def sample_batch(
    N,
    N_pop,
    a_min, a_max,
    z_min, z_max,
    r_min, r_max,
    delta, alpha,
    mix_prob,
    n_vals
):
    
    # Labor Supply and Aggregate Shocks
    n = torch.randint(0, n_vals.numel(), (N, N_pop))
    n_oth = n[:, 1:]

    empl_prob =  (n_oth == 1).float().mean(dim=1, keepdim=True)
    L_oth = n2*empl_prob + n1*(1- empl_prob)

    Z = z_min + (z_max - z_min)*torch.rand((N, 1))

    # Interest Rate and Implied Capital
    r_min_L = torch.maximum(
        alpha * torch.exp(Z) * (L_oth/a_max) ** (1-alpha) - delta,
        torch.as_tensor(r_min),
    )
    r_max_L = torch.minimum(
        alpha * torch.exp(Z) * (L_oth/a_min) ** (1-alpha) - delta,
        torch.as_tensor(r_max),
    )
    r = r_min_L + (r_max_L - r_min_L) * torch.rand((N, 1))

    k_impl = (
        alpha * torch.exp(Z) * (L_oth ** (1 - alpha)) / (r + delta)
    ) ** (1 / (1 - alpha))


    # hhld i wealth
    alpha_beta_i, beta_beta_i = beta_pars(k_impl, var_a, a_min, a_max)
    a_beta = a_min + (a_max - a_min) * Beta(alpha_beta_i, beta_beta_i).sample()
    a_uni  = a_min + (a_max-a_min)*torch.rand(N, 1)
    u = torch.rand(N, 1)
    a_i = torch.where(u < mix_prob, a_uni, a_beta)

    
    # Other households' wealth
    a_oth_cand = a_min + (a_max - a_min) * torch.rand((N, N_pop - 1))
    k_oth = a_oth_cand.mean(dim=1,keepdim=True)
    a_oth = a_oth_cand*(k_impl/k_oth)


    # Constructing Input
    inp = torch.cat((a_i, a_oth, n, Z), dim=1)
    return inp



def calc_eqm(inp):
    a_oth = inp[:, 1:N_pop]
    n_oth = inp[:, N_pop+1:2*N_pop]
    z = inp[:, 2*N_pop:2*N_pop+1]

    k = a_oth.mean(dim=1, keepdim=True)
    empl_prob =  (n_oth == 1).float().mean(dim=1, keepdim=True)
    L = n2*empl_prob + n1*(1- empl_prob)

    r = alpha * torch.exp(z) *  (L/k) ** (1 - alpha) - delta
    w = (1 - alpha) * torch.exp(z) * (k/L) ** alpha

    return r, w, k, L



In [6]:
## PDE Residuals and Shape Constraints
def pde(model_v,inp,N_pop):
    N = inp.shape[0]
    
    z = inp[:,2*N_pop:2*N_pop+1].clone()
    a = inp[:,0:N_pop].clone()
    ni = inp[:,N_pop:N_pop+1].clone()

    inp_alt = inp.detach().clone()
    inp_alt[:,N_pop]=1-inp_alt[:,N_pop]
    Wa_pred_alt = model_v(inp_alt)

    Lambda = lambdas[1]*ni + lambdas[0]*(1-ni)
    
    
    ## ----------------
    ## Equilibrium Values
    ## ----------------
    
    r,w,K,L = calc_eqm(inp)
    resid_HJBE=torch.zeros(N, 1)
    
    
    ## ----------------
    ## Idiosyncratic Shock and Consumption
    ## ----------------    
    a.requires_grad_(True)

    def model_a(a): 
        inp_temp = inp.detach().clone()
        inp_temp[:,0:N_pop] = a
        return model_v(inp_temp)

    Wa_pred = model_a(a)
    Wa_prime = torch.autograd.grad(
        Wa_pred, a, 
        create_graph=True,
        grad_outputs=torch.ones_like(Wa_pred)
    )[0]

    Wa_prime_i = Wa_prime[:,0:1]

    ai = a[:,0:1]

    C = torch.clamp_min(Wa_pred,1e-8)**(-1/gamma)
    S = r*ai + w*n2*ni + w*n1*(1-ni) - C

    mask = (ai <= a_lb).float()
    penalty = -kappa*(ai - a_lb) * mask

    resid_HJBE += (r-rho)*Wa_pred + penalty + Wa_prime_i*S + Lambda*(Wa_pred_alt - Wa_pred)
    

    ## ----------------
    ## Wealth Distribution
    ## ----------------    
    for j in range(1,N_pop):

        # Rearranging inputs for jth agent
        inp_j = inp.detach().clone()
        ind = (0,j,N_pop,N_pop +j)
        ind_switch = (j,0,N_pop+j,N_pop)
        inp_j[:,ind] = inp_j[:,ind_switch]
        
        nj = inp_j[:,N_pop:N_pop+1].clone()
        aj = inp_j[:,0:1].clone()
        
        inp_alt_j = inp_j.detach().clone()
        inp_alt_j[:,N_pop]=1-inp_alt_j[:,N_pop]

        Lambda_j = lambdas[1]*nj + lambdas[0]*(1-nj)


        # Equilibrium values for jth agent 
        rj,wj,Kj,Lj = calc_eqm(inp_j)

        # Consumption and Saving decision for jth agent
        Wa_pred_j = model_v(inp_j) 
        Wa_pred_j_alt = model_v(inp_alt_j)
        Cj = torch.clamp_min(Wa_pred_j,1e-8)**(-1/gamma)
        Sj = rj*aj + wj*n2*nj  + wj*n1*(1-nj) - Cj

        # HJBE Residual contribution from jth agent 
        resid_HJBE += Sj*Wa_prime[:,j:j+1] + Lambda_j*(Wa_pred_j_alt - Wa_pred_j)

    

    ## ----------------
    ## Aggregate Shock
    ## ----------------  
    a.requires_grad_(False)
    z.requires_grad_(True)

    def model_z(z):
        inp_temp = inp.detach().clone()
        inp_temp[:,2*N_pop:2*N_pop+1] = z
        return model_v(inp_temp)
    
    Wz_pred = model_z(z)
    Wz_prime = torch.autograd.grad(
        Wz_pred, z, 
        create_graph=True,
        grad_outputs=torch.ones_like(Wz_pred)
    )[0]
    Wzz_prime = torch.autograd.grad(
        Wz_prime, z, 
        create_graph=True,
        grad_outputs=torch.ones_like(Wz_prime)
    )[0]

    resid_HJBE += 0.5*sig2*Wzz_prime[:,0:1] + Wz_prime[:,0:1]*eta*(Zbar-z)
    z.requires_grad_(False)

    return resid_HJBE



def shape_cons_a(model_v,inp):
    N = inp.shape[0]
    r,w,K,L = calc_eqm(inp)
    
    ai = inp[:,0:1].clone()
    ai.requires_grad_(True)

    def model_a(ai): 
        inp_temp = inp.detach().clone()
        inp_temp[:,0:1] = ai
        return model_v(inp_temp)

    Wa_pred = model_a(ai)
    Wa_prime = torch.autograd.grad(
        Wa_pred, ai, 
        create_graph=True,
        grad_outputs=torch.ones_like(Wa_pred)
    )[0]

    Wa_prime_i = Wa_prime[:,0:1]
    r_eff = torch.clamp_min(r, 0.0) 
    ub = -gamma*r_eff*(Wa_pred ** ((gamma + 1) / gamma))
    
    conc_loss_a = (F.relu(Wa_prime_i-ub)**2).mean()
    ai.requires_grad_(False)
    return conc_loss_a



def shape_cons_z(model_v,inp):
    N = inp.shape[0]
    z = inp[:,2*N_pop:2*N_pop+1].clone()
    z.requires_grad_(True)

    def model_z(z):
        inp_temp = inp.detach().clone()
        inp_temp[:,2*N_pop:2*N_pop+1] = z
        return model_v(inp_temp)
    
    Wz_pred = model_z(z)
    Wz_prime = torch.autograd.grad(
        Wz_pred, z, 
        create_graph=True,
        grad_outputs=torch.ones_like(Wz_pred)
    )[0]
    conc_loss_z = (F.relu(Wz_prime)**2).mean()
    z.requires_grad_(False)
    return conc_loss_z


    

In [7]:
def cons_plot(model_v):
    N = A_grid.shape[0]
    M = N_pop - 1

    a_oth = torch.full((M,), k_ss.item(), dtype=torch.float32)   # FIXED
    n_oth = torch.cat([
        torch.full((M // 2,), 1, dtype=torch.float32),
        torch.full((M // 2,), 0, dtype=torch.float32),
    ])
    z = torch.tensor([0.0], dtype=torch.float32)

    a_oth_rep = a_oth.unsqueeze(0).repeat(a.shape[0], 1)
    n_oth_rep = n_oth.unsqueeze(0).repeat(a.shape[0], 1)
    z_rep = z.view(1,1).repeat(a.shape[0], 1)
    
    inp0 = torch.hstack((A_grid.reshape(N, 1), a_oth_rep, torch.zeros((N,1)), n_oth_rep, z_rep))
    inp1 = torch.hstack((A_grid.reshape(N, 1), a_oth_rep, torch.ones((N,1)), n_oth_rep, z_rep))

    w_pred1 = model_v(inp0)
    w_pred2 = model_v(inp1)

    C1 = torch.clamp_min(w_pred1, 1e-8)**(-1/gamma)
    C2 = torch.clamp_min(w_pred2, 1e-8)**(-1/gamma)
    return C1, C2


In [ ]:
N=2**10

inp = sample_batch(
    N, 
    N_pop, 
    a_min, a_max, 
    z_min, z_max, 
    r_min, r_max, 
    delta, alpha, 
    0.3,
    n_vals
)

r,w,K,L = calc_eqm(inp)

a_oth = inp[:,1:N_pop]
k_oth = torch.mean(a_oth,dim=1)

ai = inp[:,0:1]


r_np  = r.detach().cpu().numpy().ravel()
ai_np = ai.detach().cpu().numpy().ravel()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(r_np, bins=50)
axes[0].set_title("r")

axes[1].hist(ai_np, bins=50)
axes[1].set_title("a_i")

fig.tight_layout()
fig.savefig("hist_r_ai.png", dpi=100, bbox_inches="tight")



In [9]:
## Constructing the NN Class for Finite Agent Approximation Method
class Net_MultLayer(nn.Module):
  def __init__(self, hidden_dims):
    super(Net_MultLayer, self).__init__()

    layers=[]
    prev_dim = 2*N_pop+1 

    for h in hidden_dims:
        layers.append(nn.Linear(prev_dim,h,bias=True))
        layers.append(nn.Tanh())
        prev_dim = h 
    
    layers.append(nn.Linear(prev_dim, 1))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return F.softplus(self.model(x))

    

In [10]:
## Preparing to Train NN for Initial Value Function
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

torch.manual_seed(42)
tic     = time.perf_counter()

hidden_dims   = [64,64,64,64,64]
N       = 500

model_v0 = Net_MultLayer(hidden_dims).to(device)
optimizer = torch.optim.Adam(model_v0.parameters(), lr=1e-4)


In [ ]:
## Train the neural network
target_loss=1e-5
epoch=0


while True: 

    ## ----------------
    ## Sample
    ## ----------------
        
    inp = sample_batch(
        N, 
        N_pop, 
        a_min, a_max, 
        z_min, z_max, 
        r_min, r_max, 
        delta, alpha, 
        0,
        n_vals
    )

    a = inp[:,0:1]
    n = inp[:,N_pop:N_pop+1]
    

    ## ----------------
    ## Calculate loss on sample
    ## ----------------
    v_pred = model_v0(inp)
    n = inp[:,N_pop:N_pop+1]
    c_min = (1-alpha) * math.exp(z_max)*(a_max**(alpha))*n2 + r_max*a_max
    v_target = torch.exp(-a) + c_min**(-gamma)
    v_target = (math.log(a_max)+1e-4)*torch.ones((N,1)) - torch.log(a)
    
    # Evaluate Mean Squared Loss
    loss = F.mse_loss(v_pred,v_target)
    
    # print loss value in specific epochs
    if epoch%1000 == 0:
        print('Training loss at epoch %s: %s' % (epoch, float(loss.item())))

    ## ----------------
    ## Update parameters
    ## ----------------
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    if loss.item() < target_loss:
        print(f"Stopping early at epoch {epoch}. Loss = {loss.item()}")
        break

    epoch += 1
    

In [ ]:
M = N_pop - 1
a_oth = A_grid[dist.sample((M,))]   
idx_n = torch.randint(0, len(n_vals), (M,))
n_oth = n_vals[idx_n]              
z = torch.tensor([0.0], dtype=torch.float32)

n = torch.ones((N,1)) 


a_oth_rep = a_oth.unsqueeze(0).repeat(N, 1)  
n_oth_rep = n_oth.unsqueeze(0).repeat(N, 1)  
z_rep     = z.unsqueeze(0).repeat(N, 1)      

inp0 = torch.hstack((A_grid.reshape(N,1), a_oth_rep, torch.zeros((N,1)), n_oth_rep, z_rep))
inp1 = torch.hstack((A_grid.reshape(N,1), a_oth_rep, torch.ones((N,1)), n_oth_rep, z_rep))

v_pred1 = model_v0(inp0)
v_pred2 = model_v0(inp1)

plt.plot(A_grid.detach().cpu().numpy().reshape(-1),(v_pred1**(-1/gamma)).detach().cpu().numpy().reshape(-1))
plt.plot(A_grid.detach().cpu().numpy().reshape(-1),(v_pred2**(-1/gamma)).detach().cpu().numpy().reshape(-1))
plt.show()

plt.figure()
plt.hist(a_oth.detach().cpu().numpy().reshape(-1), bins=50)
plt.show()



In [13]:
## Preparing to Train NN for HJB
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

torch.manual_seed(42)
tic     = time.perf_counter()
N       = 1000

model_v = Net_MultLayer(hidden_dims).to(device)
model_v.load_state_dict(model_v0.state_dict())

optimizer = torch.optim.Adam(model_v.parameters(), lr=1e-4)



In [ ]:
## Train the neural network
epoch = 0

while True: 

    ## ----------------
    ## Sample
    ## ----------------    
        
    inp = sample_batch(
        N, 
        N_pop, 
        a_min, a_max, 
        z_min, z_max, 
        r_min, r_max, 
        delta, alpha, 
        0.1, 
        n_vals
    )
        
    ## ----------------
    ## Loss Function
    ## ----------------
    resid_HJBE = pde(model_v,inp,N_pop)
    conc_loss_a = shape_cons_a(model_v,inp)
    conc_loss_z = shape_cons_z(model_v,inp)

    ni = inp[:, N_pop:N_pop+1]  
    mask0 = (ni == 0).float()
    mask1 = (ni == 1).float()

    res2 = resid_HJBE.square()
    loss_pde0 = (res2*mask0).sum()/(mask0.sum()+1e-8)
    loss_pde1 = (res2*mask1).sum()/(mask1.sum()+1e-8)

    loss_pde = res2.mean()
    loss = loss_pde + conc_loss_a + conc_loss_z
    


    ## ----------------
    ## Update parameters
    ## ----------------
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_v.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()
    
    
    ## ----------------
    ## Convergence check
    ## ----------------
    mean_res = resid_HJBE.square().mean().item()
    cond_resid  = mean_res < 1e-4

    if epoch % 1000 == 0:
        clear_output(wait=True)
        C1, C2 = cons_plot(model_v)
        w_pred1 = C1**(-gamma)
        w_pred2 = C2**(-gamma)

        plt.plot(A_grid, C1.detach().cpu().numpy().reshape(-1), label="C1")
        plt.plot(A_grid, C2.detach().cpu().numpy().reshape(-1), label="C2")

        plt.legend()
        plt.show()
        plt.close()
        
    
    if epoch % 200 == 0:

        print("pde0:", loss_pde0.item(),
              "pde1:", loss_pde1.item(),
          "conc_a:", conc_loss_a.item(),
          "conc_z:", conc_loss_z.item())
        print(f"[DEBUG] mean_resid={mean_res:.4g}")
        print('Training loss at epoch %s: %s' % (epoch, float(loss.item())))


    if cond_resid: 
        print(f"\nConverged at epoch {epoch}:")
        print(f"  mean residual    = {mean_res}")
        print(f"  loss            = {loss.item()}")
        break

    epoch += 1


toc = time.perf_counter()
print(f"Completed in {toc - tic:0.4f} seconds")

In [ ]:
# Plotting Implied Consumption Functions 
C1, C2 = cons_plot(model_v)
w_pred1 = C1**(-gamma)
w_pred2 = C2**(-gamma)

plt.plot(A_grid, w_pred1.detach().cpu().numpy().reshape(-1), label="C1")
plt.plot(A_grid, w_pred2.detach().cpu().numpy().reshape(-1), label="C2")

plt.legend()
plt.show()


N = A_grid.shape[0] 
MPC = (C2[N-1] - C2[N-2])/(a[N-1] - a[N-2])

print(MPC)
